In [ ]:
# Import Library

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# To upload dataset from computer
from google.colab import files
uploaded = files.upload()
csv_file = next(iter(uploaded))
df = pd.read_csv(csv_file)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.duplicated()

In [ ]:
df.duplicated().sum()

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

In [ ]:
df.isnull()

In [ ]:
# Fill missing categorical values with the most common category, if any are missing
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isna().any():
        mode = df[col].mode()
        if not mode.empty:
            df[col] = df[col].fillna(mode.iloc[0])

In [ ]:
df.isnull().sum()

In [ ]:
# Numerical values
df.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Fill missing numeric values with each column's median, only when needed
num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

### Treatment of missing numeric values
Numeric missing values are filled with the column median only when present. This avoids replacing values unnecessarily; this dataset has no missing values.

In [ ]:
df.isnull().sum()

In [ ]:
df.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
# Outlier check
num_cols = ['year', 'selling_price', 'km_driven']
fig, axes = plt.subplots(1, len(num_cols), figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# IQR outlier check for price and kilometres driven
for col in ['selling_price', 'km_driven']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'{col}: bounds=({lower_bound:.0f}, {upper_bound:.0f}), flagged rows={len(outliers)}')

In [ ]:
# Keep valid outliers in the dataset; IQR flags values for review and does not prove they are errors.

In [ ]:
# Univariate analysis
sns.countplot(x=df['fuel'])
plt.title('Distribution of Cars by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Number of Cars')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate analysis
plt.figure(figsize=(8, 5))
sns.countplot(x=df['fuel'], hue=df['transmission'])
plt.title('Fuel Type vs Transmission')
plt.xlabel('Fuel Type')
plt.ylabel('Number of Cars')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate analysis
sns.catplot(
    x='fuel',
    hue='transmission',
    col='seller_type',
    data=df,
    kind='count',
    height=4,
    aspect=1
)
plt.show()

In [ ]:
# Categorical columns
df.select_dtypes(include='object').columns

In [ ]:
# Encode a separate copy only if numerical categories are needed later.
# Keep df unchanged so category labels remain readable for EDA.
from sklearn.preprocessing import LabelEncoder
df_encoded = df.copy()
le = LabelEncoder()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = le.fit_transform(df_encoded[col])

In [ ]:
df.info()